In [52]:
import json
import os
import pandas as pd
import torch
import sys
#torch.cuda.empty_cache()
from transformers import (
    AutoModelForCausalLM, 
    AutoTokenizer, 
    Trainer, 
    TrainingArguments, 
    DataCollatorForLanguageModeling
)
from datasets import Dataset
import tqdm as notebook_tqdm
import time

In [53]:
# Define special tokens for our math reasoning format
SPECIAL_TOKENS = {
    "bos_token": "<|startoftext|>",
    "eos_token": "<|endoftext|>",
    "sep_token": "<|sep|>",
    "calc_start": "<<",
    "calc_end": ">>",
    "answer_marker": "####"
}

In [54]:
def log_message(message):
    """Print a timestamped log message"""
    timestamp = time.strftime("%Y-%m-%d %H:%M:%S")
    print(f"[{timestamp}] {message}")
    sys.stdout.flush()  # Force output to be displayed immediately


In [55]:
def load_jsonl_data(file_path):
    """Load data from a JSONL file containing question-answer pairs"""
    examples = []
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            for line in f:
                try:
                    item = json.loads(line.strip())
                    examples.append(item)
                except json.JSONDecodeError:
                    print(f"Error parsing line: {line[:50]}...")
        
        print(f"Loaded {len(examples)} examples from {file_path}")
        return examples
    except Exception as e:
        log_message(f"Error loading data: {str(e)}")
        raise

In [56]:
def prepare_dataset(examples):
    """Convert examples into a format suitable for the Dataset object"""
    formatted_data = []
    
    for example in examples:
        # Check if example has the expected fields
        if 'question' in example and 'answer' in example:
            question = example['question']
            answer = example['answer']
            
            # Format with special tokens
            text = f"{SPECIAL_TOKENS['bos_token']}{question}{SPECIAL_TOKENS['sep_token']}{answer}{SPECIAL_TOKENS['eos_token']}"
            if isinstance(text, str) and text and SPECIAL_TOKENS["bos_token"] in text and SPECIAL_TOKENS["eos_token"] in text:
                formatted_data.append({"text":text})
            #formatted_data.append({"text": text})
    log_message(f"Created dataset with {len(formatted_data)} examples")
    return Dataset.from_dict({"text": [item["text"] for item in formatted_data]})

In [57]:
# def tokenize_data(dataset, tokenizer, max_length=512):
#     """Tokenize the dataset"""
#     log_message(f"Tokenizing dataset with {len(dataset)} examples...")
#     # def tokenize_function(examples):
#     #     return tokenizer(
#     #         examples["text"],
#     #         truncation=True,
#     #         padding="max_length",
#     #         max_length=max_length,
#     #         #return_tensors='pt'
#     #     )
#     log_message(f"Tokenization complete. Dataset size: {len(dataset)}")
#     #return dataset.map(tokenize_function, batched=True)
#     return dataset.map(lambda x:tokenizer(x['text'],truncation=True, padding='max_length',max_length=max_length), batched=False)

def tokenize_data(dataset, tokenizer, max_length=512):
    """Tokenize the dataset"""
    print("bEFORE TOKENIZING..")
    for i, example in enumerate(dataset):
        if not isinstance(example["text"], str):
            print(f"❌ Error at index {i}: Expected string, got {type(example['text'])}")
        

    

    log_message(f"Tokenizing dataset with {len(dataset)} examples...")

    def tokenize_function(examples):
        tokenized = tokenizer(
            examples["text"],
            truncation=True,
            padding="max_length",
            max_length=max_length,
            return_tensors="pt"  # Convert to PyTorch tensors
        )
        return {
            "input_ids": tokenized["input_ids"].squeeze(0).tolist(),
            "attention_mask": tokenized["attention_mask"].squeeze(0).tolist()
        }

    dataset = dataset.map(tokenize_function, batched=True)
    # Debugging: Print first 5 tokenized samples
    print("sAMPLE TOKENIZATION:")
    for i in range(5):
        print(f"✅ Sample {i} Tokenized: {dataset[i]}")
    dataset = [{'input_ids': sample['input_ids'], 'attention_mask': sample['attention_mask']} for sample in dataset]
    dataset = Dataset.from_list(dataset)
    log_message(f"Tokenization complete. Dataset size: {len(dataset)}")
    return dataset


In [58]:
def create_tokenizer(model_name, special_tokens):
    """Create and configure the tokenizer with special tokens"""
    log_message(f"Loading tokenizer for {model_name}...")
    try:
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        log_message(f"Tokenizer loaded successfully. Base vocab size: {len(tokenizer)}")
        # Add special tokens
        special_tokens_dict = {
            "bos_token": special_tokens["bos_token"],
            "eos_token": special_tokens["eos_token"],
            "sep_token": special_tokens["sep_token"],
            "additional_special_tokens": [
                special_tokens["calc_start"], 
                special_tokens["calc_end"],
                special_tokens["answer_marker"]
            ]
        }
        
        num_added_tokens=tokenizer.add_special_tokens(special_tokens_dict)
        log_message(f"Added {num_added_tokens} special tokens. New vocab size: {len(tokenizer)}")
        # Set padding token
        tokenizer.pad_token = tokenizer.eos_token
        log_message("Padding token set to EOS token")
        return tokenizer
    except Exception as e:
        log_message(f"Error creating tokenizer: {str(e)}")
        raise

In [59]:
def load_model(model_name, tokenizer_len):
    """Load the model with proper error handling and better diagnostics"""
    log_message(f"Loading model {model_name}...")
    try:
        # Check if CUDA is available
        if torch.cuda.is_available():
            log_message(f"CUDA is available. Using GPU: {torch.cuda.get_device_name(0)}")
            log_message(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
        else:
            log_message("CUDA is not available. Using CPU only.")
        
        # Try to download and load the model with a timeout
        # Note: Unfortunately, we can't easily set a timeout for model loading in transformers
        log_message("Starting model download/load (this may take several minutes)...")
        model = AutoModelForCausalLM.from_pretrained(model_name)
        log_message(f"Model {model_name} loaded successfully!")
        
        # Resize token embeddings
        model.resize_token_embeddings(tokenizer_len)
        log_message(f"Token embeddings resized to {tokenizer_len}")
        
        return model
    except Exception as e:
        log_message(f"Error loading model: {str(e)}")
        log_message("Try one of the following:")
        log_message("1. Use a smaller model like 'gpt2' instead of 'gpt2-medium'")
        log_message("2. Check your internet connection")
        log_message("3. Check if the Hugging Face model hub is accessible")
        log_message("4. Try using a local model if available")
        raise

In [60]:
def fine_tune_math_model(
    jsonl_file_path,
    model_name="gpt2",
    output_dir="/kaggle/working/math_reasoning_model",
    num_train_epochs=15,
    learning_rate=5e-5,
    warmup_ratio=0.1,
    weight_decay=0.01,
    save_steps=500,
    batch_size=4,
    gradient_accumulation_steps=4
):
    """Train a model for mathematical reasoning with explicit calculation steps"""
    try:
        # Load examples from JSONL file
        examples = load_jsonl_data(jsonl_file_path)
        
        # Create and configure tokenizer
        tokenizer = create_tokenizer(model_name, SPECIAL_TOKENS)
        
        # Prepare dataset
        dataset = prepare_dataset(examples)
        
        # Tokenize dataset
        tokenized_dataset = tokenize_data(dataset, tokenizer)
        print("column names of tokenized dataset:",tokenized_dataset.column_names)
        # Print dataset stats
        # print(f"Dataset size: {len(tokenized_dataset)}")
        # print(f"Sample example: {dataset[0]['text'][:100]}...")
        log_message(f"Dataset size: {len(tokenized_dataset)}")
        log_message(f"Sample example: {dataset[0]['text'][:100]}...")
        
        # Load model
        #model = AutoModelForCausalLM.from_pretrained(model_name)

        # Load model (this is where it might be hanging)
        model = load_model(model_name, len(tokenizer))

        
        print("model ",model_name," loaded successfully.")
        # Resize token embeddings to account for special tokens
        model.resize_token_embeddings(len(tokenizer))
        print("resized token embeddings.")

        # Configure training arguments
        log_message("Configuring training arguments...")
        # Configure training arguments
        training_args = TrainingArguments(
            output_dir=output_dir,
            evaluation_strategy="no",  # No evaluation during training
            save_strategy="steps",
            save_steps=save_steps,
            num_train_epochs=num_train_epochs,
            per_device_train_batch_size=batch_size,
            gradient_accumulation_steps=gradient_accumulation_steps,
            learning_rate=learning_rate,
            warmup_ratio=warmup_ratio,
            weight_decay=weight_decay,
            logging_dir=f"{output_dir}/logs",
            logging_steps=10,
            save_total_limit=2,
            push_to_hub=False,
            fp16=torch.cuda.is_available(),  # Use fp16 if GPU is available
            dataloader_num_workers=4,
            remove_unused_columns=False
        )

        log_message("Creating trainer...")
        # Create trainer
        trainer = Trainer(
            model=model,
            args=training_args,
            train_dataset=tokenized_dataset,
            tokenizer=tokenizer,
            data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
        )
        print("Trainer created successfully! Training model...")
        
        print("Final check:")
        for i, example in enumerate(tokenized_dataset):
            if not isinstance(example["input_ids"], list):
                print(f"❌ Error at index {i}: Expected list of token IDs, got {type(example['input_ids'])}")
            else:
                if i%500==0:
                    print(f"✅ Sample {i}: {example['input_ids'][:10]}")  # Print first 10 tokens
        
            

        log_message("Starting training...")
        # Train model
        trainer.train()
        
        # Save model and tokenizer
        trainer.save_model(output_dir)
        tokenizer.save_pretrained(output_dir)
        
        return model, tokenizer
        
    except Exception as e:
        log_message(f"Error during fine-tuning: {str(e)}")
        log_message("Training process failed.")
        raise

In [61]:
def generate_math_solution(model, tokenizer, question, max_length=512, temperature=0.7):
    """Generate a solution for a given math problem"""
    prompt = f"{SPECIAL_TOKENS['bos_token']}{question}{SPECIAL_TOKENS['sep_token']}"
    
    input_ids = tokenizer(prompt, return_tensors="pt").input_ids
    if torch.cuda.is_available():
        input_ids = input_ids.cuda()
        model = model.cuda()
    
    # Generate output
    output = model.generate(
        input_ids,
        max_length=max_length,
        temperature=temperature,
        top_p=0.95,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )
    
    generated_text = tokenizer.decode(output[0], skip_special_tokens=False)
    
    # Extract answer part (between sep_token and eos_token)
    parts = generated_text.split(SPECIAL_TOKENS['sep_token'])
    if len(parts) > 1:
        answer_part = parts[1].split(SPECIAL_TOKENS['eos_token'])[0].strip()
        return answer_part
    else:
        return "Failed to generate a proper solution."

In [62]:
def inspect_jsonl_file(file_path, num_samples=3):
    """Print a few samples from the JSONL file to understand its structure"""
    samples = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for i, line in enumerate(f):
            if i >= num_samples:
                break
            try:
                item = json.loads(line.strip())
                samples.append(item)
            except json.JSONDecodeError:
                print(f"Error parsing line {i}: {line[:50]}...")
    
    for i, sample in enumerate(samples):
        print(f"\nSample {i+1}:")
        for key, value in sample.items():
            if isinstance(value, str) and len(value) > 100:
                print(f"{key}: {value[:100]}...")
            else:
                print(f"{key}: {value}")
    
    return samples

In [63]:
import json

def validate_jsonl(file_path):
    valid_entries = []
    invalid_entries = []
    
    with open(file_path, "r", encoding="utf-8") as f:
        for line_num, line in enumerate(f, start=1):
            try:
                entry = json.loads(line.strip())
                if (
                    isinstance(entry, dict) and
                    "question" in entry and isinstance(entry["question"], str) and entry["question"].strip() and
                    "answer" in entry and isinstance(entry["answer"], str) and entry["answer"].strip()
                ):
                    valid_entries.append(entry)
                else:
                    invalid_entries.append((line_num, line.strip()))
            except json.JSONDecodeError:
                invalid_entries.append((line_num, line.strip()))
    
    if invalid_entries:
        print(f"Found {len(invalid_entries)} invalid entries in {file_path}:")
        for line_num, content in invalid_entries[:5]:  # Show only first 5 invalid lines
            print(f"Line {line_num}: {content}")
        print("... (only first 5 shown, fix all before proceeding)")
    else:
        print("All entries are correctly formatted!")
    
    return valid_entries


In [ ]:
# Main execution
if __name__ == "__main__":
    # Your specific file path
    jsonl_file_path = "/kaggle/input/math-problems-sp27/train.jsonl"
     
    # First, let's inspect the file to understand its structure
    print("Inspecting JSONL file structure...")
    inspect_jsonl_file(jsonl_file_path)
    print("Inspecting JSONL file structure...")
    validate_jsonl(jsonl_file_path)
    # Set up the output directory
    output_dir = "/kaggle/working/math_reasoning_model"
    
    # Fine-tune the model
    print(f"\nStarting model training using {jsonl_file_path}")
    print(f"Model will be saved to {output_dir}")
    
    model, tokenizer = fine_tune_math_model(
        jsonl_file_path,
        model_name="gpt2-medium",  # Using a larger model for better performance
        output_dir=output_dir,
        num_train_epochs=6,
        batch_size=2 if torch.cuda.is_available() else 1  # Smaller batch size if memory is an issue
    )
    
    # Test the model with a new question
    test_question = "James bought 3 shirts for $25 each and 2 pairs of pants for $40 each. How much did he spend in total?"
    print("\nTesting model with a new question:")
    print(f"Question: {test_question}")
    
    solution = generate_math_solution(model, tokenizer, test_question)
    print("\nGenerated solution:")
    print(solution)

Inspecting JSONL file structure...

Sample 1:
question: Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How m...
answer: Natalia sold 48/2 = <<48/2=24>>24 clips in May.
Natalia sold 48+24 = <<48+24=72>>72 clips altogether...

Sample 2:
question: Weng earns $12 an hour for babysitting. Yesterday, she just did 50 minutes of babysitting. How much ...
answer: Weng earns 12/60 = $<<12/60=0.2>>0.2 per minute.
Working 50 minutes, she earned 0.2 x 50 = $<<0.2*50...

Sample 3:
question: Betty is saving money for a new wallet which costs $100. Betty has only half of the money she needs....
answer: In the beginning, Betty has only 100 / 2 = $<<100/2=50>>50.
Betty's grandparents gave her 15 * 2 = $...
Inspecting JSONL file structure...
All entries are correctly formatted!

Starting model training using /kaggle/input/math-problems-sp27/train.jsonl
Model will be saved to /kaggle/working/math_reasoning_model
Loaded 7473 examples from /kaggle/input/math-p

Map:   0%|          | 0/7473 [00:00<?, ? examples/s]

sAMPLE TOKENIZATION:
✅ Sample 0 Tokenized: {'text': '<|startoftext|>Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?<|sep|>Natalia sold 48/2 = <<48/2=24>>24 clips in May.\nNatalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.\n#### 72<|endoftext|>', 'input_ids': [50257, 47849, 9752, 2702, 19166, 284, 4764, 286, 607, 2460, 287, 3035, 11, 290, 788, 673, 2702, 2063, 355, 867, 19166, 287, 1737, 13, 1374, 867, 19166, 750, 14393, 9752, 3677, 13318, 287, 3035, 290, 1737, 30, 50258, 47849, 9752, 2702, 4764, 14, 17, 796, 220, 16791, 2780, 14, 17, 28, 1731, 4211, 1731, 19166, 287, 1737, 13, 198, 47849, 9752, 2702, 4764, 10, 1731, 796, 220, 16791, 2780, 10, 1731, 28, 4761, 4211, 4761, 19166, 13318, 287, 3035, 290, 1737, 13, 198, 4242, 7724, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 502

/usr/local/lib/python3.10/dist-packages/transformers/training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
<ipython-input-60-a72b12753666>:70: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Trainer created successfully! Training model...
Final check:
✅ Sample 0: [50257, 47849, 9752, 2702, 19166, 284, 4764, 286, 607, 2460]
✅ Sample 500: [50257, 19585, 2826, 4929, 351, 20893, 290, 50070, 13, 679]
✅ Sample 1000: [50257, 7554, 24779, 257, 16930, 14841, 329, 720, 1270, 13]
✅ Sample 1500: [50257, 464, 4231, 2100, 78, 1641, 1816, 503, 284, 8073]
✅ Sample 2000: [50257, 51, 3301, 318, 8914, 510, 284, 2822, 257, 649]
✅ Sample 2500: [50257, 2601, 7626, 3382, 284, 787, 362, 30849, 329, 607]
✅ Sample 3000: [50257, 32, 16825, 318, 1160, 10700, 890, 13, 632, 1392]
✅ Sample 3500: [50257, 40, 1057, 1105, 4608, 287, 4101, 2431, 13, 1867]
✅ Sample 4000: [50257, 41, 11870, 290, 2940, 389, 9644, 9294, 11022, 13]
✅ Sample 4500: [50257, 3118, 2375, 16182, 468, 257, 720, 12825, 2855, 326]
✅ Sample 5000: [50257, 19591, 24779, 257, 2353, 286, 41478, 32650, 6174, 329]
✅ Sample 5500: [50257, 5990, 563, 3382, 284, 2822, 257, 649, 3660, 14043]
✅ Sample 6000: [50257, 1532, 340, 2753, 807, 1528, 329, 51

wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.


<IPython.core.display.Javascript object>